# AIE S3 — Bank Term Deposit: Modelling & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s3-bank-marketing.ipynb)

**Binary classification.** 45,211 marketing calls; predict which
clients subscribed. 11.7% positive, ranked on **F1**.

Challenge: <https://ml-arena.com/viewchallenge/184>

---

**This notebook has no code in it.** Each step is written out in
English and the cell below it is empty. You have just read the bike
notebook, which does all of this with the code showing; this is the
same protocol on a classification target, and you write it.

You already submitted here in Session 2: a logistic regression at
the default threshold, **F1 = 0.453**. That is the number to beat,
and the bar for this lab is **F1 = 0.55**.

One thing is genuinely new relative to the bike notebook, and it is
section 7. On this target the decision threshold is worth more than
any model swap in the grid.

---

## 0. Setup

Install `mlarena-sdk` — the same one-liner as every other notebook in
this course.

---

## 1. Get the data

Connect with your `mlk_user_...` key and download challenge
**184** into the working directory. Three files:
`X_train.csv`, `y_train.csv`, `X_test.csv`.

---

## 2. Read it, and name it for what it is

Read `X_train.csv` into **`X`** and `y_train.csv["prediction"]` into
**`y`** — that is everything you hold a label for. Read `X_test.csv`
into **`X_submission`**: the rows only the leaderboard can score.

Use those names. In this session `X_test` is about to mean the slice
of your own data you keep back, and two things cannot share a name.

Print the shapes, and print `y.mean()` — on a 0/1 target the mean is
the positive rate, and it is the number that decides everything that
follows.

---

## 3. Split first — before you touch a single value

Same rule as the bike notebook, **different split**, and the
difference is the point.

These rows are not a time series — the challenge's own split is a
shuffled, stratified 80/20. So use `train_test_split`, twice, with
`stratify=` and a fixed `random_state`:

1. `X` → pool (80%) + test (20%)
2. pool → train (80% of the pool) + validation (20% of the pool)

**`stratify=` is not optional on an 11.7% target.** Without it a 20%
slice can land anywhere from 10% to 13% positive depending on the
seed, and your F1 then moves with the draw rather than with the model.

Print the positive rate of all three slices and check they match.

Ask yourself why the bike notebook could not do this, and why this
notebook must not do what the bike notebook did.

---

## 4. Then transform — the minimum that works

There are no missing cells in this table, so the whole preparation is
`pd.get_dummies` on the nine text columns, plus the `reindex` onto
the training columns that keeps every matrix the same width.

Drop `id` before encoding. It is an identifier with one level per
row: hand it to `get_dummies` and you get 36,168 columns, every one
of them unknown at prediction time.

**And decide about `duration`.** It is the length of the call whose
outcome you are predicting. Whether that makes it a leak is not a
property of the column — it depends on a question this table cannot
answer for you:

> **Which decision am I modelling, and what do I know when I make it?**

*To choose who to ring*, `duration` is unusable. The call has not
happened, so the number does not exist; by the time it does, you
already know the answer and the prediction has no use left.

*To decide whether to keep a call that is already running*, it is
legitimate — elapsed seconds are known to you at that moment. But
the column as shipped is the wrong quantity for that job: it is the
**final** length of a completed call, not the elapsed length of a
live one. The honest version reshapes the data into a hazard — for a
call that has survived to *t* seconds, what is P(subscribe)? — which
you can build from this table, since a 600-second call was, at every
*t* below 600, a call still going at *t*.

And one trap under the second framing that the first does not have.
"Who do I ring" is a prediction; "do I keep talking" is an
**intervention**, and this is observational data. Long calls
correlate with sales because interested people talk longer, not
because talking longer creates interest — clients whose previous
campaign succeeded both talk longer (315s vs 259s) and subscribe far
more (64.6% vs 9.2%), and that gap survives holding call length
fixed. A policy of "stay on the phone longer" read off that
correlation would buy you longer calls, not more deposits.

Name your decision, then check every column against it. Whichever
way you go, write down which and why — that decision is worth marks;
not noticing it is not.

---

## 5. Compare models — two numbers per candidate

Fit on train, score on train **and** on validation, report both in
one table. At least these candidates:

| family | vary |
|---|---|
| `LogisticRegression(max_iter=1000)` | `C` ∈ {0.01, 0.1, 1.0}, through a `StandardScaler` |
| `RandomForestClassifier(n_estimators=200)` | `max_depth` ∈ {5, 12, None} |
| `HistGradientBoostingClassifier` | `learning_rate` ∈ {0.05, 0.1} |

Score with `f1_score`, not accuracy — always predicting `0` is 88.3%
accurate here and finds not one subscriber.

Two things to look for, and they are the reason for the table:

- one of these candidates reaches a **training F1 of 1.000**. Find it,
  and say in one sentence what its validation F1 is and why the two
  numbers are so far apart;
- the shallowest forest is the *worst* model in the table, well below
  the logistic regression. Underfitting and overfitting are both in
  this one column — name which row is which.

Then draw it: a horizontal bar chart, train and validation side by
side, as in section 5 of the bike notebook.

---

## 6. Search the hyperparameters

Take the family that won section 5 and run `GridSearchCV` over it on
the pool (train + validation).

- `cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=0)` —
  stratified for the same reason the split was, and shuffled because
  these rows have no time order to protect.
- `scoring="f1"`.

Twenty to fifty candidates is the right size. For boosting, vary
`learning_rate`, `max_iter`, `max_leaf_nodes` and `min_samples_leaf`;
for the forest, `max_depth`, `min_samples_leaf` and `max_features`.

Then read the result properly: print `best_params_`, but also sort
`cv_results_` and compare the gap between the top candidates against
`std_test_score`. If the spread across folds is larger than the gap
between your top five, you have found a region, not a winner — say so.

---

## 7. The threshold — the knob that is actually worth turning

This section has no counterpart in the bike notebook, and on this
dataset it moves the score more than everything above it combined.

`predict()` cuts `predict_proba()` at 0.5. That is a library
convention, not a statement about your problem: with 11.7% positives,
very few clients ever cross 0.5, so the model refuses to call almost
everyone and recall collapses.

Take `predict_proba(...)[:, 1]` from your best model, sweep the
threshold from 0.05 to 0.75 in steps of 0.01, and plot F1 against it
— **on the validation slice, never on the leaderboard and never on
your test slice**. Keep the argmax.

Expect the best threshold to land far below 0.5, and expect it to be
worth more than 0.07 of F1. Report, in one sentence: what the
threshold did to precision, what it did to recall, and which of the
two errors is more expensive when the action is "phone this person".

---

## 8. Spend the test set

Once. Refit your chosen model on the pool, apply your chosen
threshold, and report F1 on the test slice — the rows that have been
in no fit and no sweep.

Report it next to the validation F1 you selected on. If the test
number is materially lower, that gap is the cost of every choice you
made against validation, and it is the honest thing to write down.

---

## 9. Refit on everything, then submit

The test set has been spent, so it goes back in. Refit on all of
`X`/`y`, predict `X_submission`, apply your threshold, and write
`submission.csv` with columns `id` and `prediction`, where
`prediction` is the **integer** 0 or 1 — a probability is rejected,
not rounded for you.

Assert before uploading: one row per id in `X_submission`, ids
unique, every value in {0, 1}. Then submit and read the board.

**The bar is F1 = 0.55**, against the 0.453 your Session 2 logistic
regression scored on these same rows. A boosting model at the default
threshold clears it; a boosting model at a tuned threshold clears it
comfortably.

---

## 10. Write down what you found

Five sentences, in this notebook:

- Which candidate reached a training F1 of 1.000, and what it scored
  held out.
- Which model your grid search chose, and whether the gap to second
  place was bigger than the fold-to-fold spread.
- What threshold you kept, and what it did to precision and recall.
- What you decided about `duration`, and why.
- Your validation F1, your test F1 and your leaderboard F1, side by
  side. If they disagree, say by how much — three numbers that agree
  within noise is the result this session is asking for, and three
  that do not is a finding, not a failure.